In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("No GPU detected. Check your Colab runtime settings.")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
!pip install -U qwen-asr

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 48.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 110.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import qwen_asr

print("Qwen ASR installed successfully!")

Qwen ASR installed successfully!


In [ ]:
import torch
from qwen_asr import Qwen3ASRModel

model = Qwen3ASRModel.from_pretrained(
    "Qwen/Qwen3-ASR-1.7B",
    dtype=torch.float16,
    device_map="cuda:0",
    max_inference_batch_size=1,
    max_new_tokens=256
)

print("Qwen3-ASR-1.7B loaded successfully!")

In [ ]:
print("Model:", "Qwen3-ASR-1.7B")
print("Device:", torch.cuda.get_device_name(0))
print("Allocated GPU memory:",
      round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")

In [ ]:
from IPython.display import Audio, display
import soundfile as sf
import numpy as np

print("Audio preparation environment ready!")

In [ ]:
import time

audio_url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav"

start_time = time.time()

results = model.transcribe(
    audio=audio_url,
    language=None
)

processing_time = time.time() - start_time

print("Detected language:", results[0].language)
print("Transcription:", results[0].text)
print("Processing time:", round(processing_time, 2), "seconds")

In [ ]:
!pip install -q datasets soundfile

In [ ]:
from datasets import load_dataset

# Load English samples
english_dataset = load_dataset(
    "google/fleurs",
    "en_us",
    split="test",
    streaming=True
)

# Load Arabic samples
arabic_dataset = load_dataset(
    "google/fleurs",
    "ar_eg",
    split="test",
    streaming=True
)

print("English dataset loaded:", english_dataset)
print("Arabic dataset loaded:", arabic_dataset)

In [ ]:
english_sample = next(iter(english_dataset))
arabic_sample = next(iter(arabic_dataset))

print("English fields:", english_sample.keys())
print("English transcript:", english_sample["transcription"])

print("\nArabic fields:", arabic_sample.keys())
print("Arabic transcript:", arabic_sample["transcription"])

In [ ]:
import os
import soundfile as sf
import pandas as pd
import numpy as np
from itertools import islice

# Create dataset folders
os.makedirs("Qwen_ASR_Dataset/English", exist_ok=True)
os.makedirs("Qwen_ASR_Dataset/Arabic", exist_ok=True)

records = []

def save_samples(dataset, language, folder, count=5):

    samples = islice(iter(dataset), count)

    for i, sample in enumerate(samples, start=1):

        # Extract audio
        audio = sample["audio"]
        audio_array = np.asarray(audio["array"])
        sample_rate = audio["sampling_rate"]

        # Generate filename
        filename = f"{language}_{i}.wav"
        file_path = os.path.join(folder, filename)

        # Save audio
        sf.write(file_path, audio_array, sample_rate)

        # Calculate duration
        duration = len(audio_array) / sample_rate

        # Save information
        records.append({
            "filename": filename,
            "language": language,
            "transcript": sample["transcription"],
            "raw_transcript": sample["raw_transcription"],
            "gender": sample["gender"],
            "duration": round(duration, 2),
            "sample_rate": sample_rate,
            "file_path": file_path
        })

        print(f"Saved: {filename} | Duration: {duration:.2f}s")

# Collect 5 samples from each language
save_samples(
    english_dataset,
    "English",
    "Qwen_ASR_Dataset/English"
)

save_samples(
    arabic_dataset,
    "Arabic",
    "Qwen_ASR_Dataset/Arabic"
)

# Create metadata file
df = pd.DataFrame(records)
df.to_csv(
    "Qwen_ASR_Dataset/metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nDataset collection completed!")
print("Total samples:", len(df))

In [ ]:
print(df[[
    "filename",
    "language",
    "duration",
    "gender"
]])

print("\nTotal recordings:", len(df))
print("\nLanguage distribution:")
print(df["language"].value_counts())

In [ ]:
import time
import pandas as pd
import torch

results_list = []

for index, row in df.iterrows():

    start_time = time.time()

    audio_path = row["file_path"]

    prediction = model.transcribe(
        audio=audio_path,
        language=row["language"]
    )

    processing_time = time.time() - start_time

    results_list.append({
        "filename": row["filename"],
        "language": row["language"],
        "reference": row["transcript"],
        "prediction": prediction[0].text,
        "processing_time": round(processing_time, 2),
        "duration": row["duration"]
    })

    print(f"\nProcessed: {row['filename']}")
    print("Reference:", row["transcript"])
    print("Prediction:", prediction[0].text)
    print("Time:", round(processing_time, 2), "seconds")


results_df = pd.DataFrame(results_list)

results_df.to_csv(
    "Qwen_ASR_Dataset/predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nAll recordings processed!")

In [ ]:
!pip install -q jiwer

In [ ]:
import re
import jiwer

def normalize_text(text, language):

    text = text.lower().strip()

    # Remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    if language == "Arabic":

        # Remove Arabic diacritics
        text = re.sub(r"[\u064B-\u065F\u0670]", "", text)

        # Remove Tatweel
        text = text.replace("ـ", "")

        # Normalize Alef variations
        text = re.sub("[أإآ]", "ا", text)

        # Normalize Alef Maksura
        text = text.replace("ى", "ي")

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
wer_scores = []
cer_scores = []

for index, row in results_df.iterrows():

    reference = normalize_text(
        row["reference"],
        row["language"]
    )

    prediction = normalize_text(
        row["prediction"],
        row["language"]
    )

    wer_score = jiwer.wer(reference, prediction)
    cer_score = jiwer.cer(reference, prediction)

    wer_scores.append(wer_score)
    cer_scores.append(cer_score)

results_df["WER"] = wer_scores
results_df["CER"] = cer_scores

print(results_df[[
    "filename",
    "language",
    "WER",
    "CER"
]])

print("\nEvaluation completed!")

In [ ]:
language_results = results_df.groupby("language").agg({
    "WER": "mean",
    "CER": "mean",
    "processing_time": "mean",
    "duration": "mean"
}).reset_index()

print(language_results)

In [ ]:
results_df["RTF"] = (
    results_df["processing_time"] / results_df["duration"]
)

print("Average RTF:", round(results_df["RTF"].mean(), 3))

In [ ]:
results_df.to_csv(
    "Qwen_ASR_Dataset/final_evaluation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

language_results.to_csv(
    "Qwen_ASR_Dataset/language_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Evaluation results saved successfully!")

In [ ]:
import jiwer

for language in ["English", "Arabic"]:

    subset = results_df[
        results_df["language"] == language
    ]

    references = [
        normalize_text(text, language)
        for text in subset["reference"]
    ]

    predictions = [
        normalize_text(text, language)
        for text in subset["prediction"]
    ]

    corpus_wer = jiwer.wer(references, predictions)
    corpus_cer = jiwer.cer(references, predictions)

    print(f"\n{language} Results")
    print("Corpus WER:", round(corpus_wer, 4))
    print("Corpus CER:", round(corpus_cer, 4))


English Results
Corpus WER: 0.0385
Corpus CER: 0.0123

Arabic Results
Corpus WER: 0.1667
Corpus CER: 0.0376


In [ ]:
import numpy as np
import soundfile as sf
import os

os.makedirs("Qwen_ASR_Dataset/Noisy", exist_ok=True)

def add_noise(audio, snr_db):

    audio = audio.astype(np.float32)

    signal_power = np.mean(audio ** 2)

    noise = np.random.normal(
        0, 1, len(audio)
    ).astype(np.float32)

    noise_power = np.mean(noise ** 2)

    noise = noise * np.sqrt(
        signal_power /
        (noise_power * (10 ** (snr_db / 10)))
    )

    noisy_audio = audio + noise

    return np.clip(noisy_audio, -1, 1)


print("Noise generation function ready!")

Noise generation function ready!


In [ ]:
import soundfile as sf
import os
import pandas as pd

snr_levels = [20, 10, 5]

noise_records = []

for index, row in df.iterrows():

    # Read original audio
    audio, sr = sf.read(row["file_path"])

    for snr in snr_levels:

        # Generate noisy audio
        noisy_audio = add_noise(audio, snr)

        # Create folder
        folder = f"Qwen_ASR_Dataset/Noisy/{snr}dB"
        os.makedirs(folder, exist_ok=True)

        # Save noisy copy
        filename = f"{row['filename'].replace('.wav', '')}_{snr}dB.wav"
        file_path = os.path.join(folder, filename)

        sf.write(file_path, noisy_audio, sr)

        noise_records.append({
            "filename": filename,
            "language": row["language"],
            "reference": row["transcript"],
            "snr": snr,
            "duration": row["duration"],
            "file_path": file_path
        })

        print(f"Created: {filename}")

noisy_df = pd.DataFrame(noise_records)

print("\nNoise generation completed!")
print("Total noisy recordings:", len(noisy_df))

Created: English_1_20dB.wav
Created: English_1_10dB.wav
Created: English_1_5dB.wav
Created: English_2_20dB.wav
Created: English_2_10dB.wav
Created: English_2_5dB.wav
Created: English_3_20dB.wav
Created: English_3_10dB.wav
Created: English_3_5dB.wav
Created: English_4_20dB.wav
Created: English_4_10dB.wav
Created: English_4_5dB.wav
Created: English_5_20dB.wav
Created: English_5_10dB.wav
Created: English_5_5dB.wav
Created: Arabic_1_20dB.wav
Created: Arabic_1_10dB.wav
Created: Arabic_1_5dB.wav
Created: Arabic_2_20dB.wav
Created: Arabic_2_10dB.wav
Created: Arabic_2_5dB.wav
Created: Arabic_3_20dB.wav
Created: Arabic_3_10dB.wav
Created: Arabic_3_5dB.wav
Created: Arabic_4_20dB.wav
Created: Arabic_4_10dB.wav
Created: Arabic_4_5dB.wav
Created: Arabic_5_20dB.wav
Created: Arabic_5_10dB.wav
Created: Arabic_5_5dB.wav

Noise generation completed!
Total noisy recordings: 30


In [ ]:
print(noisy_df.groupby(
    ["language", "snr"]
).size())

print("\nTotal noisy samples:", len(noisy_df))

language  snr
Arabic    5      5
          10     5
          20     5
English   5      5
          10     5
          20     5
dtype: int64

Total noisy samples: 30


In [ ]:
import time
import pandas as pd

noisy_results = []

for index, row in noisy_df.iterrows():

    start_time = time.time()

    prediction = model.transcribe(
        audio=row["file_path"],
        language=row["language"]
    )

    processing_time = time.time() - start_time

    noisy_results.append({
        "filename": row["filename"],
        "language": row["language"],
        "snr": row["snr"],
        "reference": row["reference"],
        "prediction": prediction[0].text,
        "processing_time": round(processing_time, 2),
        "duration": row["duration"]
    })

    print(
        f"Processed: {row['filename']} | "
        f"SNR: {row['snr']} dB | "
        f"Time: {processing_time:.2f}s"
    )

noisy_results_df = pd.DataFrame(noisy_results)

noisy_results_df.to_csv(
    "Qwen_ASR_Dataset/noisy_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nAll noisy recordings processed!")
print("Total:", len(noisy_results_df))

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Processed: English_1_20dB.wav | SNR: 20 dB | Time: 3.00s


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Processed: English_1_10dB.wav | SNR: 10 dB | Time: 2.98s


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Processed: English_1_5dB.wav | SNR: 5 dB | Time: 2.20s


... (48 outputs left out)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Processed: Arabic_5_20dB.wav | SNR: 20 dB | Time: 1.31s


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Processed: Arabic_5_10dB.wav | SNR: 10 dB | Time: 1.33s
Processed: Arabic_5_5dB.wav | SNR: 5 dB | Time: 1.19s

All noisy recordings processed!
Total: 30


In [ ]:
import jiwer

noisy_results_df["WER"] = noisy_results_df.apply(
    lambda row: jiwer.wer(
        normalize_text(row["reference"], row["language"]),
        normalize_text(row["prediction"], row["language"])
    ),
    axis=1
)

noisy_results_df["CER"] = noisy_results_df.apply(
    lambda row: jiwer.cer(
        normalize_text(row["reference"], row["language"]),
        normalize_text(row["prediction"], row["language"])
    ),
    axis=1
)

noise_comparison = noisy_results_df.groupby(
    ["language", "snr"]
).agg({
    "WER": "mean",
    "CER": "mean",
    "processing_time": "mean"
}).reset_index()

print(noise_comparison.to_string(index=False))

language  snr      WER      CER  processing_time
  Arabic    5 0.219573 0.088079            1.848
  Arabic   10 0.174936 0.067505            2.046
  Arabic   20 0.154936 0.048205            2.038
 English    5 0.042907 0.018021            2.016
 English   10 0.074486 0.048661            2.410
 English   20 0.042907 0.014317            2.340


In [ ]:
clean_comparison = results_df.groupby("language").agg({
    "WER": "mean",
    "CER": "mean",
    "processing_time": "mean"
}).reset_index()

clean_comparison["condition"] = "Clean"
clean_comparison["snr"] = "Clean"

noisy_comparison = noise_comparison.copy()
noisy_comparison["condition"] = "Noisy"
noisy_comparison["snr"] = (
    noisy_comparison["snr"].astype(str) + " dB"
)

final_comparison = pd.concat(
    [clean_comparison, noisy_comparison],
    ignore_index=True
)

print(final_comparison.to_string(index=False))

final_comparison.to_csv(
    "Qwen_ASR_Dataset/clean_vs_noisy_comparison.csv",
    index=False
)

language      WER      CER  processing_time condition   snr
  Arabic 0.154936 0.034192            2.002     Clean Clean
 English 0.042907 0.014317            1.192     Clean Clean
  Arabic 0.219573 0.088079            1.848     Noisy  5 dB
  Arabic 0.174936 0.067505            2.046     Noisy 10 dB
  Arabic 0.154936 0.048205            2.038     Noisy 20 dB
 English 0.042907 0.018021            2.016     Noisy  5 dB
 English 0.074486 0.048661            2.410     Noisy 10 dB
 English 0.042907 0.014317            2.340     Noisy 20 dB


In [ ]:
import pandas as pd

duration_analysis = results_df.copy()

# Divide recordings into short and long groups
median_duration = duration_analysis["duration"].median()

duration_analysis["duration_group"] = (
    duration_analysis["duration"]
    .apply(lambda x: "Short" if x <= median_duration else "Long")
)

# Compare performance
duration_results = duration_analysis.groupby(
    ["language", "duration_group"]
).agg({
    "duration": "mean",
    "processing_time": "mean",
    "WER": "mean",
    "CER": "mean"
}).reset_index()

print(duration_results.to_string(index=False))

language duration_group  duration  processing_time      WER      CER
  Arabic           Long 13.626667         2.083333 0.140171 0.034386
  Arabic          Short  8.250000         1.880000 0.177083 0.033901
 English           Long 11.010000         1.580000 0.026316 0.005051
 English          Short  6.280000         0.933333 0.053968 0.020495


In [ ]:
import torch
import time

# Synchronize GPU
torch.cuda.synchronize()

# Record memory before inference
baseline_memory = torch.cuda.memory_allocated(0)

# Reset peak memory statistics
torch.cuda.reset_peak_memory_stats(0)

# Run inference on all clean recordings
start_time = time.time()

for index, row in df.iterrows():

    prediction = model.transcribe(
        audio=row["file_path"],
        language=row["language"]
    )

torch.cuda.synchronize()

total_time = time.time() - start_time

# Get peak GPU memory
peak_allocated = torch.cuda.max_memory_allocated(0) / 1024**3
peak_reserved = torch.cuda.max_memory_reserved(0) / 1024**3

baseline_gb = baseline_memory / 1024**3

total_gpu = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
)

print("GPU:", torch.cuda.get_device_name(0))
print("Total GPU memory:", round(total_gpu, 2), "GB")
print("Baseline allocated:", round(baseline_gb, 2), "GB")
print("Peak allocated:", round(peak_allocated, 2), "GB")
print("Peak reserved:", round(peak_reserved, 2), "GB")
print("Inference time for 10 samples:", round(total_time, 2), "seconds")

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


GPU: Tesla T4
Total GPU memory: 14.56 GB
Baseline allocated: 3.81 GB
Peak allocated: 3.96 GB
Peak reserved: 4.46 GB
Inference time for 10 samples: 16.04 seconds


In [ ]:
duration_results.to_csv(
    "Qwen_ASR_Dataset/duration_analysis.csv",
    index=False
)

print("Duration analysis saved!")

Duration analysis saved!


In [ ]:
hardware_results = pd.DataFrame([{
    "Model": "Qwen3-ASR-1.7B",
    "GPU": torch.cuda.get_device_name(0),
    "Total_GPU_Memory_GB": 14.56,
    "Baseline_Allocated_GB": 3.81,
    "Peak_Allocated_GB": 3.96,
    "Peak_Reserved_GB": 4.46,
    "Inference_Time_10_Samples": 16.04,
    "Average_Time_Per_Sample": 16.04 / 10
}])

hardware_results.to_csv(
    "Qwen_ASR_Dataset/hardware_results.csv",
    index=False
)

print(hardware_results.to_string(index=False))
print("\nHardware results saved!")

         Model      GPU  Total_GPU_Memory_GB  Baseline_Allocated_GB  Peak_Allocated_GB  Peak_Reserved_GB  Inference_Time_10_Samples  Average_Time_Per_Sample
Qwen3-ASR-1.7B Tesla T4                14.56                   3.81               3.96              4.46                      16.04                    1.604

Hardware results saved!
